# 09 Java 核心与 JVM 入门

> 覆盖范围：Java 语言基础（面向对象/集合/泛型/线程）、**JVM 运行时数据区**（堆/栈/方法区）、**类加载机制**（双亲委派）、**垃圾回收 GC**（可达性/分代/收集器）、内存泄漏与调优（OOM/堆转储）、JVM 参数、Java 与 Go/Python 对比、面试高频追问。
> 使用方式：先读**原理讲解**（JVM 如何运行 class、GC 为什么分代、双亲委派防什么），再看 **Java 代码示例**（默写），最后用 Python 模拟核心语义验证，自测面试追问。
> 注：本 notebook 内核为 Python 3；Java/JVM 片段用于阅读与默写，对照实验用 Python 模拟以保持可执行验证。

---

## 1. Java 与 JVM 是什么

- **Java**：静态类型、面向对象、**一次编译到处运行**（Write Once, Run Anywhere）的后端主力语言；
- **JVM（Java Virtual Machine）**：解释/编译（JIT）字节码的虚拟机——`.java` → `javac` 编译 → `.class` 字节码 → JVM 运行；
- 生态：Spring 全家桶、Hadoop、Kafka、Android；大厂后端（银行/电商/高并发）主流；
- 对比：比 Go 成熟（生态/框架），比 Python 快（JIT + 静态类型），内存管理全自动（GC）但调优复杂。

## 2. 基础语法速览

```java
public class Hello {                       // 类名与文件名一致
    public static void main(String[] args) {   // 程序入口
        int a = 1;
        String s = "hello";
        System.out.println(a + s);

        // 面向对象
        Dog d = new Dog("旺财");
        System.out.println(d.speak());
    }
}

class Dog {
    private String name;                    // 封装：private 字段 + getter
    public Dog(String name) { this.name = name; }   // 构造器
    public String speak() { return "woof from " + name; }
}
```

**三支柱**：

- **封装**：private 字段 + public 方法（信息隐藏）；
- **继承**：`extends`（单继承），`super` 调父类；
- **多态**：接口 `implements` + 方法覆写（override）——父类引用指向子类对象，运行时按实际类型调用。

## 3. 集合框架（面试高频）

**Collection（单列）** 与 **Map（键值对）**：

| 接口 | 实现 | 底层 | 特点 |
| --- | --- | --- | --- |
| List | **ArrayList** | 动态数组 | 随机访问 O(1)，尾部 O(1)，中间 O(n) |
| List | **LinkedList** | 双向链表 | 中间插入 O(1)（需指针），随机访问 O(n) |
| Set | **HashSet** | 哈希表（基于 HashMap） | 去重，O(1)，无序 |
| Set | TreeSet | 红黑树 | 有序，O(log n) |
| Map | **HashMap** | 哈希表（数组+链表+红黑树） | O(1)，**线程不安全** |
| Map | ConcurrentHashMap | 分段锁/CAS+同步 | 线程安全，高并发用 |
| Map | TreeMap | 红黑树 | 有序 |

**HashMap 原理（必考）**：数组 + 链表（JDK8 起桶内链表长度 >8 转红黑树）；`hash(key)` 扰动后与容量-1 求与定位；冲突用链表/树；**扩容**：负载因子 0.75，超阈值翻倍并 rehash。**线程不安全**：并发 put 可能丢数据/死循环（JDK7 头插法环），并发场景用 ConcurrentHashMap。

## 4. 线程与 synchronized

```java
// 创建线程两种方式
class T extends Thread { public void run() {...} }          // 方式1：继承
// 方式2（推荐）：实现 Runnable，解耦任务与线程
Runnable r = () -> System.out.println("task");
new Thread(r).start();

// 同步：synchronized（互斥锁）
synchronized void inc() { count++; }          // 锁 this
synchronized (obj) { ... }                    // 锁对象

// volatile：可见性（不保证原子性）
volatile boolean running = true;
// 线程池（替代手动 new Thread）
ExecutorService pool = Executors.newFixedThreadPool(4);
pool.submit(task);
```

**synchronized vs volatile vs Lock**：volatile 只保证**可见性**（读最新值）；synchronized 保证**可见性+原子性**（互斥）；`java.util.concurrent.locks.ReentrantLock` 功能更强（可中断/超时/公平锁），需手动 unlock。

## 5. JVM 运行时数据区（必考）

```
┌────────────────────────────────────────┐
│  线程共享：                              │
│  ┌──────────────┐  ┌───────────────┐   │
│  │ 堆 Heap       │  │ 方法区 Metaspace│  │  ← 对象实例 / 类元信息+静态变量+常量池
│  └──────────────┘  └───────────────┘   │
│  线程私有：                              │
│  ┌─────────┐┌──────────┐┌──────────┐   │
│  │ 虚拟机栈 ││本地方法栈 ││程序计数器PC│   │  ← 栈帧(局部变量/操作数栈) / native / 当前指令
│  └─────────┘└──────────┘└──────────┘   │
└────────────────────────────────────────┘
```

| 区域 | 线程 | 存什么 | 异常 |
| --- | --- | --- | --- |
| **堆 Heap** | 共享 | 所有对象实例、数组 | OOM（堆溢出） |
| **方法区**（JDK8=Metaspace） | 共享 | 类信息、静态变量、常量池 | OOM（元空间） |
| **虚拟机栈 Stack** | 私有 | 栈帧：局部变量、操作数栈 | **StackOverflowError**（递归太深） |
| **程序计数器 PC** | 私有 | 当前执行指令地址 | 无 |

**栈帧**：每个方法调用压一个栈帧（局部变量表 + 操作数栈 + 返回地址）；方法返回出栈——所以局部变量随方法结束自动释放，而对象在堆里由 GC 管。

## 6. 类加载机制：双亲委派（必考）

类加载：**加载 → 验证 → 准备 → 解析 → 初始化**。

**双亲委派模型**：类加载请求**先交给父加载器**，父加载不了才自己加载：

```
Bootstrap（引导，JVM 核心类，如 java.lang）
   ↑ 委派
Extension（扩展，JDK 扩展目录）
   ↑ 委派
Application（应用，classpath 下的类）
   ↑ 委派
用户自定义
```

- **为什么要双亲委派**：① **安全**——防止自己写个 `java.lang.String` 冒充核心类被加载（核心类只能由 Bootstrap 加载）；② **避免重复加载**——同名的类只加载一次。
- 打破双亲委派：实现 `ClassLoader.loadClass` 覆写（如 Tomcat、SPI、热部署）。

## 7. 垃圾回收 GC（最核心）

### 7.1 判断对象可回收：可达性分析

- 从 **GC Roots**（栈上局部变量引用的对象、静态变量、常量）出发，**可达的对象存活**，不可达即垃圾；
- 不是引用计数（无法解决循环引用）；
- 引用分强/软/弱/虚四种（软引用做缓存、弱引用做 WeakHashMap）。

### 7.2 分代收集（新生代/老年代）

**对象大部分朝生夕死** → 分而治之：

```
新生代 Eden + S0 + S1（8:1:1）——大多数对象在这出生即死
  │ 熬过若干次 Minor GC/幸存者年龄达阈值
  ▼
老年代 —— 大对象兜底 + 晋升对象，Major/Full GC 频率低
```

- **Minor GC（新生代）**：Eden 满触发，复制算法，快；
- **Full GC（整个堆）**：老年代满触发，慢（要 Stop-The-World）；
- **STW（Stop The World）**：GC 时暂停业务线程——调优就是减少 STW。

### 7.3 收集器选型（背名字即可）

Serial（单线程，客户端）→ Parallel（吞吐优先，JDK8 默认）→ CMS（并发标记清除，响应优先）→ **G1（分块+可预测停顿，JDK9+ 默认）** → ZGC/Shenandoah（超低停顿）。

## 8. 内存问题排查

```bash
java -Xms512m -Xmx2g -XX:+HeapDumpOnOutOfMemoryError App   # 堆大小 + OOM 自动转储
jps          # 查 Java 进程 PID
jstat -gc <pid>   # 看 GC 频率/耗时
jmap -dump:format=b,file=heap.hprof <pid>   # 堆转储
jstack <pid>  # 线程栈（查死锁/卡顿）
jvisualvm / MAT   # 图形化分析堆转储（查大对象/泄漏）
```

**常见 OOM**：堆溢出（new 太多）、栈溢出（递归）、元空间溢出（反射/类加载过多）、直接内存溢出。

## 9. synchronized 与锁升级（高频）

`synchronized` 底层是对象头里的 **Monitor**（监视器锁），JDK6 后做了**锁升级**优化（不是一上来就上重量级锁）：

```
无锁 → 偏向锁（只有一个线程反复获取，记录线程 ID，几乎零开销）
     → 轻量级锁（有竞争时 CAS 自旋，多轮后…）
     → 重量级锁（阻塞等待，线程挂起/唤醒，开销大）
```

- **偏向锁**：同线程重入不再同步（Mark Word 存线程 ID），撤销才有代价；
- **轻量级锁**：CAS 抢锁，抢不到先**自旋**（忙等），避免线程切换；
- **重量级锁**：进入 Monitor 阻塞队列，涉及用户态/内核态切换（慢）。

## 10. volatile 与 CAS（并发基石）

**volatile**（轻量级同步）：

- **可见性**：写 volatile 变量立即刷主存，读从主存读（不走线程缓存）；
- **禁止重排序**：内存屏障保证读写有序——**DCL 单例的经典用法**；
- ❌ **不保证原子性**：`volatile int i; i++` 仍是三步（读-算-写），多线程会丢更新。

**CAS（Compare And Swap）**：乐观锁思路——比较当前值，等于期望值才更新，否则重试：

- `java.util.concurrent.atomic`（AtomicInteger 等）基于 CAS + 自旋；
- **ABA 问题**：值被 A→B→A 中间被改过但最终值相同，CAS 误判未变——用版本号（AtomicStampedReference）解决；
- 应用：`ConcurrentHashMap` 的节点插入、AQS 的锁状态更新。

## 11. 线程池 ThreadPoolExecutor（高频）

```java
new ThreadPoolExecutor(
    corePoolSize,        // 核心线程数（常驻）
    maximumPoolSize,     // 最大线程数
    keepAliveTime,       // 非核心线程空闲存活时间
    TimeUnit.SECONDS,
    workQueue,           // 任务队列（ArrayBlockingQueue/LinkedBlockingQueue/…）
    threadFactory,       // 线程工厂（命名/守护）
    handler              // 拒绝策略
);
```

**任务提交流程**（必背）：

```
提交任务
  ├─ 线程数 < corePoolSize      → 新建核心线程执行
  ├─ 线程数 ≥ core 且队列没满    → 放进 workQueue 排队
  ├─ 队列满且线程数 < maximum    → 新建非核心线程执行
  └─ 队列满且线程数 = maximum    → 执行拒绝策略
```

**拒绝策略（4 种）**：`AbortPolicy`（抛异常，默认）/ `CallerRunsPolicy`（调用者执行）/ `DiscardPolicy`（丢弃）/ `DiscardOldestPolicy`（丢最旧）。

> **生产建议**：不要用 `Executors.newFixedThreadPool` 等快捷方法（`LinkedBlockingQueue` 无界队列会堆积任务、OOM）——手动指定队列大小与拒绝策略。

## 12. 集合体系与 String（速记）

| 问题 | 答案 |
| --- | --- |
| ArrayList 扩容 | 初始 10，满后扩 **1.5 倍**（Arrays.copyOf） |
| LinkedList | 双向链表，中间插删 O(1)；随机访问 O(n) |
| HashSet 底层 | HashMap 的 key 集合（去重靠 hashCode+equals） |
| HashMap 键要求 | 重写 `equals` 必须重写 `hashCode`（否则同键进不同桶） |
| String 不可变 | final char[]；`+` 拼接会建新对象，循环用 StringBuilder |
| String vs StringBuilder | String 不可变线程安全；StringBuilder 可变不保证并发安全但快 |
| 常量池 | `"a"+"b"` 编译期优化进常量池；`new String("ab")` 不走常量池 |
| Integer 缓存 | `-128~127` 缓存在常量池，`==` 比较需注意（`valueOf` 用到缓存） |


## 核心概念

### (a) 🗂️ Java：从「JVM 内存」到「GC 与 HashMap」的主线

Java 面试三大件：**JVM 运行时数据区**（对象放哪、栈帧怎么压）、**垃圾回收**（怎么判断死对象、分代怎么回收）、**HashMap 结构**（数组+链表+红黑树怎么装数据）。把这三条线串起来就能覆盖高频题。

```mermaid
flowchart TB
    A["🧠 JVM 运行时数据区<br/>栈帧 / 堆 / 方法区"]
    A --> B["🗑️ 垃圾回收 GC<br/>可达性分析 / 分代收集"]
    B --> C["📦 HashMap<br/>数组+链表+红黑树"]
    C --> D["⚡ 扩容与冲突<br/>负载因子 / 重哈希"]
    D --> E["🎯 面试 30 秒速答<br/>高频追问闭环"]
```

- **JVM 运行时数据区**：线程私有（虚拟机栈/本地方法栈/程序计数器）vs 线程共享（堆/方法区）；方法调用压栈帧，递归太深就栈溢出。
- **垃圾回收 GC**：从 GC Roots 做**可达性分析**找存活对象，不可达的才回收；分代收集（新生代频繁 Minor GC、老年代 Major GC）是为了按对象年龄分级处理。
- **HashMap**：数组 + 链表（>8 且容量足够时树化红黑树），`hash & (n-1)` 定位桶；链表存冲突元素。
- **扩容与冲突**：负载因子默认 0.75，超过就扩容翻倍并重新散列（重哈希）；好 hashCode 减少冲突、差 hashCode 全挤一个桶退化成链表/树。
- **面试 30 秒速答**：能把内存区、GC 流程、HashMap 存取与扩容说成一条连贯的故事，Java 八股基本过关。

## 本节要解决的问题

| 要解决的问题 | 对应内容 |
|---|---|
| `new` 出来的对象到底住哪？ | JVM 运行时数据区（堆/栈/方法区） |
| 递归太深为什么 StackOverflow？ | 栈帧压栈与溢出 |
| 谁负责把没人要的对象清走？ | 垃圾回收与可达性分析 |
| 新生代老年代为什么分开？ | 分代收集与收集器 |
| HashMap 链表什么时候变红黑树？ | 树化阈值与扩容 |
| 多线程改共享变量为什么错？ | 线程安全（synchronized/volatile） |
| 内存一直涨怎么排查？ | 内存泄漏与 OOM 调优 |
| 线程池怎么配才不 OOM？ | 队列与拒绝策略 |

## 故事引入

第一次启动 Java 服务，你发现它「慢半拍」：JVM 要先把类加载、初始化、预热。跑起来后它又像「自动保洁的仓库」——你只管 `new`，JVM 悄悄把没人用的对象回收掉（GC），你几乎感觉不到。

但并发一高、对象一泄漏，GC 忙不过来：`top` 里 CPU 飙高、日志刷 `OutOfMemoryError`，服务直接雪崩。这时候能救你的只有**懂 JVM**：对象住哪个区、GC 怎么判断谁该死、HashMap 在背后怎么树化成红黑树、线程池队列为什么不能配无界。这篇把 JVM 的内存布局、GC 机制与高频容器/并发考点讲透，下次内存问题你就有抓手了。

## 深入原理

### ① 运行时数据区：栈帧与堆

**直觉**：Java 程序跑在 JVM 里，内存分「线程私有/线程共享」两块——**虚拟机栈**（每调一次方法压一个栈帧：局部变量表/操作数栈/返回地址，返回就弹栈）、**堆**（几乎一切对象）、**方法区**（JDK8 后为 Metaspace，存类信息/常量池）。

**要点**：`new` 的对象都在堆上（栈里只存引用）；递归太深→栈帧压不满→StackOverflowError；堆放不下→OutOfMemoryError。

**面试怎么问**：「方法里的局部变量存在哪？」——栈帧的局部变量表；但对象本体在堆。

![JVM 类加载、运行时数据区与执行引擎](images/java_jvm.svg)

> 📌 JVM 运行时数据区：栈/堆/方法区，栈帧随方法调用压栈弹栈，对象住堆由 GC 管辖

### ② 类加载：双亲委派为什么稳

**直觉**：类「用到才加载」：加载 → 验证 → 准备 → 解析 → 初始化。加载器有层级：**Bootstrap**（JDK 核心类）→ **Platform/Extension** → **Application**（classpath）。**双亲委派** = 加载请求先交给父加载器，父加载不成才自己加载。

**要点**：好处：核心类（`java.lang.String`）永远源自 JDK，防止被自定义类冒充；同路径类只加载一次。打破场景：`Thread.currentThread().getContextClassLoader()`（SPI）、Tomcat 每应用独立类加载器、热部署。

**面试怎么问**：「为什么要双亲委派？」——安全（防核心类被篡改）+ 一致性（不重复加载）。

### ③ 垃圾回收：可达性分析与引用分类

**直觉**：GC 判断对象该死有两种思路——**引用计数**（有循环引用就死锁）与**可达性分析**（从 GC Roots 出发按引用链遍历，够不到的就是垃圾；Java 用后者）。GC Roots：栈帧局部变量、静态变量、常量池引用、JNI 引用。

**要点**：引用分四级：强（正常引用，绝不回收）、软（内存不够才回收，做缓存）、弱（下次 GC 就回收，`WeakHashMap`）、虚（跟踪回收）。不可达对象也不是立即回收——要经过 `finalize()` 机会（但不推荐依赖）。

**面试怎么问**：「两个对象互相引用、但没被其他人引用，会回收吗？」——会：从 GC Roots 不可达即为垃圾，可达性分析天然解决循环引用。

![HashMap 哈希、桶索引与扩容](images/java_hashmap.svg)

> 📌 HashMap：数组桶 + 链表冲突，链表 ≥8 且桶 ≥64 树化红黑树，负载因子 0.75 扩容

### ④ 分代收集与收集器

**直觉**：绝大多数对象「朝生夕死」→ 堆分代：**新生代**（Eden + 两个 Survivor，8:1:1）放新对象，Minor GC 用**复制算法**（存活少、搬得快）；熬过阈值的对象晋升**老年代**，Major/Full GC 用**标记-清除/整理**（存活多、STW 长）。

**要点**：STW（Stop The World）= GC 期间暂停业务线程；调优 = 尽量走 Minor GC、减少 Full GC。收集器谱系：Serial（单线程）→ Parallel（吞吐优先，JDK8 默认）→ CMS（并发低停顿）→ **G1**（JDK9+ 默认，Region 分块、可预测停顿）→ ZGC（毫秒级停顿）。

**面试怎么问**：「为什么新生代用复制算法？」——存活率低，复制成本低且无内存碎片；老年代存活率高，复制浪费，用标记-整理。

![GC 标记、复制与老年代回收](images/java_gc.svg)

> 📌 分代收集：新生代复制算法频繁 Minor GC，存活对象晋升老年代标记-整理

### ⑤ HashMap：哈希、冲突、树化、扩容

**直觉**：HashMap 是「**数组 + 链表 + 红黑树**」：`put` 先算 `hash(key)`，再 `与运算` 定位桶下标（`hash & (n-1)` 等价取模且更快）；冲突的 key 挂链表（尾部插入）。JDK8 起链表过长（≥8）且桶数 ≥64 时树化为红黑树（查询 O(n)→O(log n)）。

**要点**：负载因子默认 0.75——`size > 容量 × 0.75` 触发扩容：新容量翻倍、全部元素重哈希（rehash）。重写 `equals` 必须重写 `hashCode`（否则同 key 进不同桶）。线程不安全：并发用 `ConcurrentHashMap`（CAS + synchronized 锁桶）。

**面试怎么问**：「为什么默认负载因子是 0.75？」——时间/空间折中：太小浪费内存，太大冲突多、退化链表。

### ⑥ 并发正确性：synchronized / volatile / 原子类 / 线程池

**直觉**：并发三性问题：**可见性**（线程缓存）、**原子性**（读改写被打断）、**有序性**（指令重排）。`synchronized` 三性齐管；`volatile` 只管可见性 + 禁重排（不保证原子）；`AtomicInteger` 用 CAS 解决原子；线程池解决「频繁创建线程」的浪费。

**要点**：锁升级：无锁 → 偏向锁（单线程）→ 轻量级锁（CAS 自旋）→ 重量级锁（阻塞）；长短任务并发要注意 `volatile` 不能替代锁；线程池七参数：core → 队列 → max → 拒绝策略（Abort/CallerRuns/Discard/DiscardOldest）。

**面试怎么问**：「`volatile` 能保证 `i++` 正确吗？」——不能，`i++` 是读-改-写三步，要用 `AtomicInteger` 或加锁；`volatile` 适合状态标志位、DCL 双重检查的 instance 字段。

In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
# 模拟 JVM 运行时数据区：栈帧压栈/出栈 + 栈溢出 + 堆对象（GC 管辖）

class JVMFrame:                       # 栈帧
    def __init__(self, method, locals_):
        self.method = method          # 方法名
        self.locals = locals_         # 局部变量表

class JVMStack:                       # 虚拟机栈（线程私有）
    def __init__(self, max_depth=10):
        self.frames = []
        self.max_depth = max_depth

    def push(self, frame):
        if len(self.frames) >= self.max_depth:
            raise RuntimeError('StackOverflowError: 栈深度超限')
        self.frames.append(frame)

    def pop(self):
        return self.frames.pop()

    def depth(self):
        return len(self.frames)

# 递归调用：每层一个栈帧
stack = JVMStack(max_depth=5)
try:
    for i in range(10):
        stack.push(JVMFrame(f'fact({i})', {'n': i}))
except RuntimeError as e:
    print(e)
    print('此时栈深:', stack.depth(), '(已达上限 5)')

# 正常返回：出栈
while stack.depth() > 0:
    f = stack.pop()
    print(f'出栈 {f.method}，释放局部变量 {f.locals}')
print('方法返回后栈帧自动释放（对比：堆里的对象等 GC）')
assert stack.depth() == 0

In [ ]:
# 模拟可达性分析 GC + 分代收集（对象晋升）

class GCObject:
    def __init__(self, obj_id, gen='young'):
        self.id = obj_id
        self.gen = gen                 # young / old
        self.refs = set()              # 指向其他对象的引用
        self.age = 0                   # 熬过 GC 次数

class MiniJVM:
    def __init__(self):
        self.objects = {}
        self.roots = set()             # GC Roots（栈/静态变量引用的对象）
        self.young = set()
        self.old = set()

    def alloc(self, oid):
        o = GCObject(oid)
        self.objects[oid] = o
        self.young.add(oid)
        return o

    def gc(self):
        """可达性分析：从 roots 广度遍历，不可达对象回收"""
        alive = set()
        stack = list(self.roots)
        while stack:
            oid = stack.pop()
            if oid in alive or oid not in self.objects:
                continue
            alive.add(oid)
            stack.extend(self.objects[oid].refs)   # 沿引用链
        dead = set(self.objects) - alive
        for oid in dead:
            self.objects.pop(oid)
            self.young.discard(oid)
            self.old.discard(oid)
        # 分代：young 存活者年龄+1，达 15 晋升老年代
        for oid in list(self.young):
            o = self.objects[oid]
            o.age += 1
            if o.age >= 3:                      # 模拟晋升阈值
                self.young.discard(oid)
                self.old.add(oid)
                o.gen = 'old'
        return dead

jvm = MiniJVM()
# main 引用的对象（存活）；局部变量离开后 roots 里移除（模拟）
a = jvm.alloc('a')
b = jvm.alloc('b')
c = jvm.alloc('c')
a.refs.add('b')
jvm.roots = {'a'}                # 只有 a 及其引用链可达
dead = jvm.gc()
print('第一次 GC 回收:', dead, '(c 不可达被回收，b 经 a 可达存活)')
assert 'c' not in jvm.objects and 'b' in jvm.objects

# 循环引用也能被回收（可达性分析 vs 引用计数）
x = jvm.alloc('x'); y = jvm.alloc('y')
x.refs.add('y'); y.refs.add('x')   # 互相引用，但 roots 中无
dead2 = jvm.gc()
print('循环引用 x,y 被回收:', dead2, ' <- 可达性分析能处理循环引用')
assert 'x' not in jvm.objects

# 晋升
jvm.roots = {'a'}
for _ in range(4): jvm.gc()
print('多次 GC 后:', 'a' in jvm.old, 'b' in jvm.old, ' <- 熬过阈值的对象晋升老年代')
assert 'a' in jvm.old

In [ ]:
# 模拟 HashMap 结构（数组+链表+红黑树阈值）与扩容/冲突

class HashMap:
    """极简 HashMap：数组 + 链表（Python 列表模拟桶）"""
    def __init__(self, capacity=4, load_factor=0.75):
        self.capacity = capacity
        self.load_factor = load_factor
        self.buckets = [[] for _ in range(capacity)]   # 每个桶是链表
        self.size = 0

    def _hash(self, key):
        return hash(key) % self.capacity               # 简化：直接取模

    def put(self, key, value):
        h = self._hash(key)
        bucket = self.buckets[h]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (k, value)                # 键已存在：覆盖
                return
        bucket.append((key, value))                   # 冲突：链表尾插
        self.size += 1
        if self.size / self.capacity > self.load_factor:
            self._resize()                            # 超负载因子：扩容

    def get(self, key):
        for k, v in self.buckets[self._hash(key)]:
            if k == key:
                return v
        return None

    def _resize(self):
        old = self.buckets
        self.capacity *= 2                            # 翻倍
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        for bucket in old:
            for k, v in bucket:
                self.put(k, v)                        # rehash
        print(f'触发扩容: capacity={self.capacity}')

m = HashMap(capacity=4)
for i in range(10):
    m.put(f'k{i}', i)
print('size=', m.size, 'capacity=', m.capacity)
print('get k5 =', m.get('k5'), '| get 不存在 =', m.get('x'))
chain_len = max(len(b) for b in m.buckets)
print('最长桶链长:', chain_len, ' <- 冲突集中时链表变长（JDK8 超8转红黑树）')
assert m.get('k5') == 5 and m.size == 10
print('\nHashMap 关键点：哈希定位 O(1) -> 冲突链表 -> 负载因子 0.75 触发翻倍扩容 rehash')
print('Java 的 HashMap 线程不安全；并发用 ConcurrentHashMap')

## 13. 面试 30 秒速答

- **JVM 数据区**：堆（对象，共享）+ 方法区/Metaspace（类信息）+ 虚拟机栈（栈帧，私有）+ PC；栈溢出 StackOverflow、堆溢出 OOM；
- **GC 判断**：可达性分析（从 GC Roots 出发），不是引用计数（能处理循环引用）；
- **分代**：新生代（Eden+S0+S1，复制算法，Minor GC 快）+ 老年代（Major/Full GC，STW 长）；对象熬过阈值晋升；
- **收集器**：Serial → Parallel（JDK8 默认，吞吐优先）→ CMS → **G1（JDK9+ 默认）** → ZGC；
- **双亲委派**：先父后子，防核心类被冒充 + 防重复加载；
- **HashMap**：数组+链表+红黑树，负载因子 0.75，扩容翻倍 rehash；线程不安全，并发用 ConcurrentHashMap；
- **synchronized vs volatile**：volatile 只保证可见性；synchronized 可见性+原子性；
- **OOM 排查**：jps → jstat 看 GC → jmap 堆转储 → MAT 分析。
- **锁升级**：偏向锁（单线程）→ 轻量级锁（CAS 自旋）→ 重量级锁（阻塞）；
- **volatile**：可见性 + 禁重排，不保证原子性（i++ 丢了）；CAS 乐观锁小心 ABA；
- **线程池七参数**：core→队列→maximum，满了拒绝；别用无界队列 Executors（OOM）；
- **String**：不可变、常量池、循环拼接用 StringBuilder；Integer 缓存 -128~127。

## 14. 自测清单（面试追问）

- [ ] 堆/栈/方法区各存什么？谁线程共享谁线程私有？StackOverflow 与 OOM 怎么分别触发？
- [ ] 可达性分析是什么？GC Roots 有哪些？为什么不用引用计数？
- [ ] 为什么分代收集？Minor GC 与 Full GC 区别？STW 是什么？
- [ ] 手写：一个对象从创建到晋升老年代的完整路径（Eden→S0→S1→晋升）。
- [ ] 各种垃圾收集器差异？（Serial/Parallel/CMS/G1/ZGC 适用场景）
- [ ] 双亲委派模型是什么？为什么要这样设计？什么场景要打破？
- [ ] HashMap 底层结构？为什么链表超 8 转红黑树？为什么负载因子 0.75？为什么线程不安全？
- [ ] ArrayList vs LinkedList 怎么选？HashSet vs TreeSet？
- [ ] synchronized / volatile / ReentrantLock 区别？volatile 能保证原子性吗？
- [ ] 手写：单例模式（双检锁 DCL + volatile）；生产者消费者（wait/notify 或 BlockingQueue）。
- [ ] 内存泄漏场景？（静态集合持有对象、未关闭资源、ThreadLocal、监听器未移除）
- [ ] 线上 CPU 100% / 频繁 Full GC 怎么排查？（top/jstack/jstat/jmap）
- [ ] 手写 JVM 参数：堆 2G、OOM 转储、打印 GC 日志（-Xms2g -Xmx2g -XX:+HeapDumpOnOutOfMemoryError）。
- [ ] synchronized 锁升级过程？（偏向→轻量→重量）为什么要有偏向锁？
- [ ] volatile 能保证原子性吗？为什么要禁止重排序？CAS 的 ABA 问题是什么？
- [ ] 线程池参数怎么配？任务提交后按什么顺序执行？四种拒绝策略各自适用？
- [ ] 手写：线程池 + CountDownLatch 等待全部任务完成。
- [ ] String / StringBuilder / StringBuffer 区别？Integer 的 == 比较要注意什么？
- [ ] HashMap 为什么要求 equals 和 hashCode 一起重写？

> ✅ 全部能回答 + 能默写代码，本课通过。

## 面试考点（Q&A）

**Q1：StackOverflowError 和 OutOfMemoryError 分别什么场景？**
A：前者是递归/栈帧过深导致虚拟机栈满；后者是堆（或元空间）无法分配新对象。💡 追问：怎么排查 OOM？——调 `-Xmx`、用 `jmap`/`jstat` 看堆、MAT 分析 dump。

**Q2：可达性分析的 GC Roots 有哪些？**
A：栈帧局部变量、静态变量、JNI 引用、常量池引用。💡 追问：循环引用的两个对象会被回收吗？——会，只要从 Roots 不可达（这正是不用引用计数的原因）。

**Q3：为什么新生代用复制算法、老年代用标记-整理？**
A：新生代存活率低，复制便宜且无碎片；老年代存活率高，复制浪费，标记-整理合适。💡 追问：Minor GC 和 Full GC 区别？——前者只回收新生代（快），后者全堆（慢、STW 长）。

**Q4：HashMap 什么时候树化、什么时候退化？**
A：链表长度 ≥8 且桶数 ≥64 树化红黑树；长度 ≤6 退化回链表。💡 追问：负载因子 0.75 的意义？——时间/空间的折中，超过就扩容翻倍重哈希。

**Q5：volatile 能保证 n++ 正确吗？**
A：不能——volatile 只保证可见性和有序性，不保证原子性；要用 AtomicInteger 或 synchronized。💡 追问：volatile 适合什么场景？——状态标志位、双重检查锁的 instance 字段。

**Q6：双亲委派模型怎么运作、解决什么问题？**
A：加载请求自下而上交给父加载器，父加载器失败才自己加载，保证核心类始终由 Bootstrap 加载、不被篡改。💡 追问：怎么打破双亲委派？——重写 loadClass 自定义加载（如 Tomcat 的 WebAppClassLoader、SPI）。

## 小结与下一步

Java 八股核心是「**内存怎么分、对象怎么活、垃圾怎么清**」：运行时数据区（栈/堆/方法区）→ 类加载（双亲委派）→ GC（可达性+分代）→ 容器（HashMap 树化）→ 并发（synchronized/volatile/原子）→ 排查（OOM/线程池）。把这条链讲成连贯故事，Java 面试就稳了。

**下一步**：回到 14-具身智能 00-机器人学基础——把语言基础收一收，进入机器人位姿与运动学的硬核世界（按岗位需要也可先复习 14 章主线）。